# Análise de dados Industriais

#### Importações

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime


In [ ]:
# Função para detectar outliers
def listar_outliers(df, coluna):
    """
    Retorna um novo DataFrame contendo apenas os outliers da coluna especificada.
    Utiliza o método do Intervalo Interquartil (IQR).
    """
    # 1. Calcular os quartis e o IQR
    Q1 = df[coluna].quantile(0.25)
    Q3 = df[coluna].quantile(0.75)
    IQR = Q3 - Q1

    # 2. Definir os limites inferior e superior
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    # 3. Filtrar o DataFrame para retornar apenas os outliers
    outliers = df[(df[coluna] < limite_inferior) | (df[coluna] > limite_superior)]

    return outliers

In [ ]:
# dicionario para mapear o codigo pela maquina
cod_maquina = {
    'MAQ-01': 'TORNO CNC 01',
    'MAQ-02': 'TORNO CNC 02',
    'MAQ-03': 'FRESADORA 01',
    'MAQ-04': 'FRESADORA 02',
    'MAQ-05': 'PRENSA HIDRÁULICA 01',
    'MAQ-06': 'PRENSA HIDRÁULICA 02',
    'MAQ-07': 'SOLDA ROBOTIZADA 01',
    'MAQ-08': 'SOLDA ROBOTIZADA 02',
    'MAQ-09': 'INJETORA PLÁSTICA 01',
    'MAQ-10': 'EXTRUSORA 01',
    'MAQ-11': 'INDETERMINADO'
}

### Extração e Tratamento dos dados sensores.json

In [186]:
df_sensores = pd.read_json('/content/sensores.json')
df_medicoes = pd.DataFrame(dict(df_sensores['medicoes'])).T

df_sensores = df_sensores.drop(columns=['medicoes'], axis=1)

df_sensores = pd.concat([df_sensores, df_medicoes], axis=1)

# O df_sensores apresentou inconsistencia na quantidade de registro da serie 'temperaturas'
# Etapa 1 verificar a quantidade de registos nulos
# print(df_sensores.isna().sum())
# Verificado 6 registros nulos
# Etapa 2 substituir os valores nulos pela mediana
df_sensores['temperatura'] = df_sensores['temperatura'].fillna(df_sensores['temperatura'].median())

# Verificar duplicados
# print(f'Duplicados: {df_sensores.duplicated().sum()}')
# verificado 4 duplicados
# Etapa 3 excluir duplicados
df_sensores = df_sensores.drop_duplicates()
# converter para datetime e padronizar o formato para futuro merge
df_sensores['timestamp'] = pd.to_datetime(df_sensores['timestamp']).dt.strftime('%Y-%m-%d')
# renomear a coluna timestamp para data
df_sensores = df_sensores.rename(columns={'timestamp': 'data'})
df_sensores['data'] = pd.to_datetime(df_sensores['data'])

# Inserir uma nova coluna mapeada pelo codigo usando o dicionario cod_maquina
df_sensores.insert(1, 'maquina', df_sensores['id_maquina'].map(cod_maquina))

# Padronizando os valores da coluna status
df_sensores['status'] = df_sensores['status'].str.strip().str.lower()

# Valores negativos na pressaõ substituidos pela mediana
mediana_pressao = df_sensores[df_sensores['pressao'] > 0]['pressao'].median()
df_sensores.loc[df_sensores['pressao'] <= 0, 'pressao'] = mediana_pressao

# Foi verificado duplicidade de leituras de sensores na mesma data de leitura
df_sensores.duplicated(subset=['id_maquina', 'data'], keep=False).sum()
# Soluçao deletar os duplicados, mantendo a primeira ocorrencia
df_sensores = df_sensores.drop_duplicates(subset=['id_maquina', 'data'], keep='first')

df_sensores.sort_values('data').head(15)

df_sensores['status'].unique()


array(['operando', 'falha'], dtype=object)

### Extracao e Tratamento dos dados producao.csv

In [ ]:
df_producao = pd.read_csv('/content/producao.csv')

# Etapa 1 Verificar dados nulos
# print(df_producao.isna().sum())
# identificado valores nulos na 'quantidade_produzida', 'tempo_producao' e 'operador'
# Substituir valores nulos de 'quantidade_produzida', 'tempo_producao' pela mediana
df_producao['quantidade_produzida'] = df_producao['quantidade_produzida']\
                .fillna(df_producao['quantidade_produzida'].median())
df_producao['tempo_producao'] = df_producao['tempo_producao']\
                .fillna(df_producao['tempo_producao'].median())
# Substituir valores nulos de 'operador' pela mediana
df_producao['operador'] = df_producao['operador']\
                .fillna(df_producao['operador'].mode()[0])

# Etapa 2 Verificar Duplicados
# print(df_producao.duplicated().sum())
# Localizado 5 valores duplciados
df_producao = df_producao.drop_duplicates()
# Convertendo data para datetime e padronizando o formato
df_producao['data'] = pd.to_datetime(df_producao['data'], yearfirst=True, format='mixed')

# Etapa 3 Padronização dos nomes das Máquinas
# print(df_producao['maquina'].unique())
'''Removendo espaços do inicio e final com strip, alterando para caixa alta com upper \
e substituindo espaços duplo por espaço simples'''
df_producao['maquina'] = df_producao['maquina'].str.strip().str.upper().str.replace('  ', ' ')
'''Nos locais nos numeros 1 adicionei um zero e depois replace 00 por 0 para que
o padrão seja exemplo: INJETORA PLÁSTICA 01'''
df_producao['maquina'] = df_producao['maquina'].str[:-1] + '0' + df_producao['maquina'].str[-1]
df_producao['maquina'] = df_producao['maquina'].str.replace('00', '0')

# print('\n Após padronização do nome')
# print(df_producao['maquina'].unique())

# Foi identificado valores iguais a zero em tempo_producao
# print(df_producao[df_producao['tempo_producao'] == 0][['maquina', 'tempo_producao']])
# Substitui valores 0 pela mediana do tempo_producao e arendodar para duas casas decimais
df_producao['tempo_producao'] = df_producao['tempo_producao'].replace(0, df_producao['tempo_producao'].median()).round(2)

# padronizando os turnos
# print(df_producao['turno'].unique())
df_producao['turno'] = df_producao['turno'].str.strip().str.lower()
# print(df_producao['turno'].unique())

# Alterando o tipo float para inteiro da coluna quantidade_produzida
df_producao['quantidade_produzida'] = df_producao['quantidade_produzida'].astype('int')

# quantidade_produzida com valores negativos, substituido pela mediana
mediana_qtd_produzida = df_producao[df_producao['quantidade_produzida'] > 0]['quantidade_produzida'].median()
df_producao.loc[df_producao['quantidade_produzida'] < 0, 'quantidade_produzida'] = mediana_qtd_produzida

# id_producao duplicado 3 registros
# df_producao['id_producao'].duplicated().sum()
# deletando a duplicidade e mantendo a primeira ocorrencia
df_producao = df_producao.drop_duplicates(subset='id_producao', keep='first')

df_producao


,id_producao,data,maquina,produto,quantidade_produzida,quantidade_planejada,tempo_producao,operador,turno
0,PR0081,2025-03-07,EXTRUSORA 01,Suporte Metálico,522,600,399.00,Carlos Souza,tarde
1,PR0100,2025-09-03,PRENSA HIDRÁULICA 02,Carcaça Plástica,653,700,399.50,Marcos Pereira,tarde
2,PR0340,2025-03-30,EXTRUSORA 01,Componente Hidráulico,295,300,431.00,André Nascimento,tarde
3,PR0332,2025-03-30,FRESADORA 01,Eixo Automotivo,445,480,406.00,Marcos Pereira,manhã
4,PR0089,2025-08-03,SOLDA ROBOTIZADA 02,Engrenagem Industrial,325,350,476.00,Marcos Pereira,manhã
...,...,...,...,...,...,...,...,...,...
343,PR0066,2025-03-06,PRENSA HIDRÁULICA 01,Engrenagem Industrial,292,350,441.00,André Nascimento,tarde
344,PR0126,2025-03-11,SOLDA ROBOTIZADA 02,Suporte Metálico,512,600,402.00,Carlos Souza,tarde
345,PR0252,2025-03-23,FRESADORA 01,Eixo Automotivo,406,480,434.00,André Nascimento,tarde
346,PR0057,2025-05-03,SOLDA ROBOTIZADA 02,Suporte Metálico,560,600,2704.66,Fernanda Lima,noite


### Extracao e Tratamento dos dados manutencao.xlsx

In [ ]:
df_manutencao = pd.read_excel('manutencao.xlsx')

# df_manutencao.isna().sum()
# Valores Nulos pela mediana
df_manutencao['tempo_parada'] = df_manutencao['tempo_parada'].fillna(df_manutencao['tempo_parada'].median())
df_manutencao['tempo_parada'] = df_manutencao['tempo_parada'].astype('int')
# df_manutencao.isna().sum()

# Verifica duplicados
# df_manutencao.duplicated().sum()

# Padronização Nomes das Maquinas
# print(df_manutencao['maquina'].unique())
'''Removendo espaços do inicio e final com strip, alterando para caixa alta com upper \
e substituindo espaços duplo por espaço simples'''
df_manutencao['maquina'] = df_manutencao['maquina'].str.strip().str.upper().str.replace('  ', ' ')
'''Nos locais nos numeros 1 adicionei um zero e depois replace 00 por 0 para que
o padrão seja exemplo: INJETORA PLÁSTICA 01'''
df_manutencao['maquina'] = df_manutencao['maquina'].str[:-1] + '0' + df_manutencao['maquina'].str[-1]
df_manutencao['maquina'] = df_manutencao['maquina'].str.replace('00', '0')
# print(df_manutencao['maquina'].unique())

# Padronizando data
df_manutencao['data'] = pd.to_datetime(df_manutencao['data'], format='mixed', yearfirst=True)

# Padronização dos tipos
# print(df_manutencao['tipo_manutencao'].unique())
df_manutencao['tipo_manutencao'] = df_manutencao['tipo_manutencao'].str.strip().str.upper().str.replace('  ', ' ')
# print(df_manutencao['tipo_manutencao'].unique())

# id_manutencao duplicado identificado
# df_manutencao[df_manutencao.duplicated(subset='id_manutencao', keep=False)]
# Deletando as duplicicadades mantendo a pŕimeira ocorrencia
df_manutencao = df_manutencao.drop_duplicates(subset=['id_manutencao'], keep='first')

# Ajustando valores negativos do tempo_parada
mediana_tempo_parada = df_manutencao[df_manutencao['tempo_parada'] > 0]['tempo_parada'].median()
df_manutencao.loc[df_manutencao['tempo_parada'] < 0, 'tempo_parada'] = mediana_tempo_parada

# Ajustando valores negativo do custo_manutancao pela mediana geral
mediana_custo_manut = df_manutencao[df_manutencao['custo_manutencao'] > 0]['custo_manutencao'].median()
df_manutencao.loc[df_manutencao['custo_manutencao'] < 0, 'custo_manutencao'] = mediana_custo_manut

df_manutencao



,id_manutencao,maquina,data,tipo_manutencao,motivo,tempo_parada,custo_manutencao,tecnico
0,MAN0066,SOLDA ROBOTIZADA 02,2025-06-03,CORRETIVA,Substituição de rolamento,98,824.57,Eduardo Martins
1,MAN0017,TORNO CNC 02,2025-03-29,PREVENTIVA,Ajuste de calibração,104,733.04,Vanessa Teixeira
2,MAN0010,TORNO CNC 02,2025-03-02,PREVENTIVA,Ajuste de calibração,117,774.66,Renata Dias
3,MAN0005,TORNO CNC 01,2025-03-08,PREVENTIVA,Substituição de rolamento,47,720.72,Eduardo Martins
4,MAN0087,EXTRUSORA 01,2025-03-14,PREVENTIVA,Lubrificação programada,112,553.13,Vanessa Teixeira
...,...,...,...,...,...,...,...,...
88,MAN0046,PRENSA HIDRÁULICA 02,2025-03-01,PREVENTIVA,Limpeza de filtros,75,983.41,Vanessa Teixeira
89,MAN0058,SOLDA ROBOTIZADA 01,2025-03-13,PREVENTIVA,Revisão elétrica,75,742.73,Sérgio Barbosa
90,MAN0023,FRESADORA 01,2025-03-16,PREVENTIVA,Troca de peça de desgaste,117,291.68,Vanessa Teixeira
91,MAN0007,TORNO CNC 01,2025-03-17,CORRETIVA,Troca de peça de desgaste,58,652.54,Renata Dias


### Extracao e Tratamento dos dados qualidade.parquet

In [ ]:
df_qualidade = pd.read_parquet('/content/qualidade.parquet')

# Verificando e atualizando valores nulos para a mediana
df_qualidade.isna().sum()
df_qualidade['quantidade_aprovada'] = df_qualidade['quantidade_aprovada'].fillna(df_qualidade['quantidade_aprovada'].median())

# Verificando e deletando duplicados
# df_qualidade.duplicated().sum()

# Padronização Nomes das Maquinas
# print(df_qualidade['maquina'].unique())
'''Removendo espaços do inicio e final com strip, alterando para caixa alta com upper \
e substituindo espaços duplo por espaço simples'''
df_qualidade['maquina'] = df_qualidade['maquina'].str.strip().str.upper().str.replace('  ', ' ')
'''Nos locais nos numeros 1 adicionei um zero e depois replace 00 por 0 para que
o padrão seja exemplo: INJETORA PLÁSTICA 01'''
df_qualidade['maquina'] = df_qualidade['maquina'].str[:-1] + '0' + df_qualidade['maquina'].str[-1]
df_qualidade['maquina'] = df_qualidade['maquina'].str.replace('00', '0')
# print(df_qualidade['maquina'].unique())

df_qualidade['quantidade_aprovada'] = df_qualidade['quantidade_aprovada'].astype('int')
df_qualidade['meta_qualidade'] = df_qualidade['meta_qualidade'].astype('int')

# Visualisar valores acima de 100 do indice_qualidade e substituir pela mediana
# print(df_qualidade[df_qualidade['indice_qualidade'] > 100])
# Calculo da mediana sobre os indices <= 100
mediana_iq = df_qualidade[df_qualidade['indice_qualidade'] <= 100]['indice_qualidade'].median()
# Substituição dos valores inconsistentes pela mediana
df_qualidade.loc[df_qualidade['indice_qualidade'] > 100, 'indice_qualidade'] = mediana_iq

# id_producao duplicado verificado
# df_qualidade.duplicated(subset='id_producao').sum() # verificação de duplicados na Serie id_producao
df_qualidade = df_qualidade.drop_duplicates(subset='id_producao', keep='first')

df_qualidade


,id_producao,maquina,produto,quantidade_inspecionada,quantidade_aprovada,quantidade_rejeitada,indice_qualidade,meta_qualidade
0,PR0222,TORNO CNC 01,Suporte Metálico,256,254,2,99.22,95
1,PR0236,PRENSA HIDRÁULICA 02,Engrenagem Industrial,88,87,1,98.86,95
2,PR0322,PRENSA HIDRÁULICA 01,Componente Hidráulico,153,136,7,95.10,95
3,PR0163,PRENSA HIDRÁULICA 02,Engrenagem Industrial,117,110,7,94.02,95
4,PR0266,SOLDA ROBOTIZADA 02,Suporte Metálico,258,225,33,87.21,95
...,...,...,...,...,...,...,...,...
231,PR0160,PRENSA HIDRÁULICA 01,Suporte Metálico,92,86,6,93.48,95
232,PR0233,FRESADORA 02,Engrenagem Industrial,227,213,14,93.83,95
233,PR0277,SOLDA ROBOTIZADA 02,Engrenagem Industrial,99,79,20,79.80,95
234,PR0070,TORNO CNC 01,Carcaça Plástica,162,149,13,91.98,95


### Merge df_producao -> df_sensores


In [ ]:
df_producao.info()
df_sensores.info()

<class 'pandas.core.frame.DataFrame'>
Index: 340 entries, 0 to 347
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id_producao           340 non-null    object        
 1   data                  340 non-null    datetime64[ns]
 2   maquina               340 non-null    object        
 3   produto               340 non-null    object        
 4   quantidade_produzida  340 non-null    int64         
 5   quantidade_planejada  340 non-null    int64         
 6   tempo_producao        340 non-null    float64       
 7   operador              340 non-null    object        
 8   turno                 340 non-null    object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(5)
memory usage: 26.6+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 310 entries, 0 to 587
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           -------

In [ ]:
df_merge_01 = pd.merge(
    df_producao,
    df_sensores,
    on=['maquina', 'data'],
    how='inner'
)

df_merge_01

,id_producao,data,maquina,produto,quantidade_produzida,quantidade_planejada,tempo_producao,operador,turno,id_maquina,status,consumo_energia,temperatura,vibracao,pressao,velocidade
0,PR0081,2025-03-07,EXTRUSORA 01,Suporte Metálico,522,600,399.0,Carlos Souza,tarde,MAQ-10,operando,14.58,65.7,1.72,5.03,1207.1
1,PR0340,2025-03-30,EXTRUSORA 01,Componente Hidráulico,295,300,431.0,André Nascimento,tarde,MAQ-10,operando,10.47,59.5,2.52,5.50,1141.6
2,PR0332,2025-03-30,FRESADORA 01,Eixo Automotivo,445,480,406.0,Marcos Pereira,manhã,MAQ-03,operando,10.75,70.2,2.51,4.75,1086.6
3,PR0014,2025-03-02,FRESADORA 01,Engrenagem Industrial,301,350,449.0,Roberto Alves,noite,MAQ-03,operando,13.38,69.9,1.93,4.75,1001.2
4,PR0198,2025-03-17,PRENSA HIDRÁULICA 02,Eixo Automotivo,480,480,408.0,Carlos Souza,tarde,MAQ-06,operando,9.73,62.4,1.80,4.45,1300.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307,PR0060,2025-03-05,EXTRUSORA 01,Carcaça Plástica,698,700,378.0,Camila Rocha,tarde,MAQ-10,operando,10.55,62.6,1.57,4.93,1187.3
308,PR0023,2025-03-03,TORNO CNC 02,Componente Hidráulico,252,300,382.0,Fernanda Lima,noite,MAQ-02,operando,9.84,63.4,2.42,5.02,1296.7
309,PR0066,2025-03-06,PRENSA HIDRÁULICA 01,Engrenagem Industrial,292,350,441.0,André Nascimento,tarde,MAQ-05,operando,12.45,59.3,1.43,5.49,1288.3
310,PR0126,2025-03-11,SOLDA ROBOTIZADA 02,Suporte Metálico,512,600,402.0,Carlos Souza,tarde,MAQ-08,operando,11.47,75.8,1.48,4.66,1317.9


### Merge df_merge_01 -> df_manutencao

In [ ]:
df_merge_01.info()
df_manutencao.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 312 entries, 0 to 311
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id_producao           312 non-null    object        
 1   data                  312 non-null    datetime64[ns]
 2   maquina               312 non-null    object        
 3   produto               312 non-null    object        
 4   quantidade_produzida  312 non-null    int64         
 5   quantidade_planejada  312 non-null    int64         
 6   tempo_producao        312 non-null    float64       
 7   operador              312 non-null    object        
 8   turno                 312 non-null    object        
 9   id_maquina            312 non-null    object        
 10  status                312 non-null    object        
 11  consumo_energia       312 non-null    float64       
 12  temperatura           312 non-null    float64       
 13  vibracao            

In [ ]:
df_merge_02 = pd.merge(
    df_merge_01,
    df_manutencao,
    on=['maquina', 'data'],
    how='left'
)

df_merge_02

,id_producao,data,maquina,produto,quantidade_produzida,quantidade_planejada,tempo_producao,operador,turno,id_maquina,...,temperatura,vibracao,pressao,velocidade,id_manutencao,tipo_manutencao,motivo,tempo_parada,custo_manutencao,tecnico
0,PR0081,2025-03-07,EXTRUSORA 01,Suporte Metálico,522,600,399.0,Carlos Souza,tarde,MAQ-10,...,65.7,1.72,5.03,1207.1,NaN,NaN,NaN,NaN,NaN,NaN
1,PR0340,2025-03-30,EXTRUSORA 01,Componente Hidráulico,295,300,431.0,André Nascimento,tarde,MAQ-10,...,59.5,2.52,5.50,1141.6,NaN,NaN,NaN,NaN,NaN,NaN
2,PR0332,2025-03-30,FRESADORA 01,Eixo Automotivo,445,480,406.0,Marcos Pereira,manhã,MAQ-03,...,70.2,2.51,4.75,1086.6,NaN,NaN,NaN,NaN,NaN,NaN
3,PR0014,2025-03-02,FRESADORA 01,Engrenagem Industrial,301,350,449.0,Roberto Alves,noite,MAQ-03,...,69.9,1.93,4.75,1001.2,NaN,NaN,NaN,NaN,NaN,NaN
4,PR0198,2025-03-17,PRENSA HIDRÁULICA 02,Eixo Automotivo,480,480,408.0,Carlos Souza,tarde,MAQ-06,...,62.4,1.80,4.45,1300.0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307,PR0060,2025-03-05,EXTRUSORA 01,Carcaça Plástica,698,700,378.0,Camila Rocha,tarde,MAQ-10,...,62.6,1.57,4.93,1187.3,NaN,NaN,NaN,NaN,NaN,NaN
308,PR0023,2025-03-03,TORNO CNC 02,Componente Hidráulico,252,300,382.0,Fernanda Lima,noite,MAQ-02,...,63.4,2.42,5.02,1296.7,NaN,NaN,NaN,NaN,NaN,NaN
309,PR0066,2025-03-06,PRENSA HIDRÁULICA 01,Engrenagem Industrial,292,350,441.0,André Nascimento,tarde,MAQ-05,...,59.3,1.43,5.49,1288.3,NaN,NaN,NaN,NaN,NaN,NaN
310,PR0126,2025-03-11,SOLDA ROBOTIZADA 02,Suporte Metálico,512,600,402.0,Carlos Souza,tarde,MAQ-08,...,75.8,1.48,4.66,1317.9,NaN,NaN,NaN,NaN,NaN,NaN


### Merge df_prod_qua_manutencao -> df_qualidade= df_base

In [ ]:
df_merge_02.info()
df_qualidade.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 312 entries, 0 to 311
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id_producao           312 non-null    object        
 1   data                  312 non-null    datetime64[ns]
 2   maquina               312 non-null    object        
 3   produto               312 non-null    object        
 4   quantidade_produzida  312 non-null    int64         
 5   quantidade_planejada  312 non-null    int64         
 6   tempo_producao        312 non-null    float64       
 7   operador              312 non-null    object        
 8   turno                 312 non-null    object        
 9   id_maquina            312 non-null    object        
 10  status                312 non-null    object        
 11  consumo_energia       312 non-null    float64       
 12  temperatura           312 non-null    float64       
 13  vibracao            

In [ ]:
df_base = pd.merge(
    df_merge_02,
    df_qualidade,
    on=['id_producao', 'maquina', 'produto'],
    how='left'
)
df_base.info()
df_base.to_csv('dados_tratados.csv')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 312 entries, 0 to 311
Data columns (total 27 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   id_producao              312 non-null    object        
 1   data                     312 non-null    datetime64[ns]
 2   maquina                  312 non-null    object        
 3   produto                  312 non-null    object        
 4   quantidade_produzida     312 non-null    int64         
 5   quantidade_planejada     312 non-null    int64         
 6   tempo_producao           312 non-null    float64       
 7   operador                 312 non-null    object        
 8   turno                    312 non-null    object        
 9   id_maquina               312 non-null    object        
 10  status                   312 non-null    object        
 11  consumo_energia          312 non-null    float64       
 12  temperatura              312 non-nul

---
## Engenharia de Dados - Enriquecimento dos dados
---

In [187]:
# Criar uma variável eficiência
df_base['eficiencia'] = df_base['quantidade_produzida'] / df_base['quantidade_planejada']

# Criar uma variável meta_cumprida
df_base['meta_cumprida'] = df_base['quantidade_produzida'] >= df_base['quantidade_planejada']

# Criar variável taxa_rejeicao
df_base['taxa_rejeicao'] = df_base['quantidade_rejeitada'] / df_base['quantidade_inspecionada']

# Cria variável consumo_energia_medio
df_base['consumo_energia_medio'] = df_base.groupby(['maquina'])['consumo_energia'].transform('mean').round(2)

# Cria variável consumo_por_unidade
df_base['consumo_por_unidade'] = df_base['consumo_energia_medio'] / df_base['quantidade_produzida']

# Criar variável custo_manutencao_dia para cada máquina
# df_base['custo_manutencao_dia'] = df_base.groupby(['maquina'])[]
df_base['custo_manutencao_dia'] = df_base.groupby('maquina')['custo_manutencao'].transform(
    lambda x: x.sum() / len(df_base['data'].unique()))

# Criar variável custo_manutencao_por_unidade
df_base['custo_manutencao_por_unidade'] = df_base['custo_manutencao_dia'] / df_base['quantidade_produzida']

df_base.to_parquet('dados_industriais_final.parquet')
df_base.to_csv('dados_industriais_final.csv')
df_base.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 312 entries, 0 to 311
Data columns (total 34 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   id_producao                   312 non-null    object        
 1   data                          312 non-null    datetime64[ns]
 2   maquina                       312 non-null    object        
 3   produto                       312 non-null    object        
 4   quantidade_produzida          312 non-null    int64         
 5   quantidade_planejada          312 non-null    int64         
 6   tempo_producao                312 non-null    float64       
 7   operador                      312 non-null    object        
 8   turno                         312 non-null    object        
 9   id_maquina                    312 non-null    object        
 10  status                        312 non-null    object        
 11  consumo_energia               31

---
### Análise Exploratória
---

##### 1. Qual máquina produz mais?


In [ ]:
idx_max = df_base.groupby(['maquina'])['quantidade_produzida'].sum().idxmax()
max_producao = df_base.groupby(['maquina'])['quantidade_produzida'].sum().max()
print(f'1. Qual máquina produz mais? \n "{idx_max}" produziu {max_producao} peças \n')
print(df_base.groupby(['maquina'])['quantidade_produzida'].sum().sort_values(ascending=False).reset_index().head(3))

1. Qual máquina produz mais? 
 "SOLDA ROBOTIZADA 02" produziu 16018 peças 

               maquina  quantidade_produzida
0  SOLDA ROBOTIZADA 02                 16018
1         TORNO CNC 02                 15799
2         EXTRUSORA 01                 15057


##### 2. Qual máquina apresenta menor eficiência?


In [ ]:
idx_min_eficiencia = df_base.groupby(['maquina'])['eficiencia'].mean().idxmin()
min_eficiencia_media = df_base.groupby(['maquina'])['eficiencia'].mean().min()
print(f'"{idx_min_eficiencia}" {min_eficiencia_media:.4f}\n')

print(df_base.groupby(['maquina'])['eficiencia'].mean().sort_values().reset_index().head(3))


"FRESADORA 01" 0.9016

                maquina  eficiencia
0          FRESADORA 01    0.901594
1  PRENSA HIDRÁULICA 01    0.907788
2   SOLDA ROBOTIZADA 02    0.909284


##### 3. Qual máquina possui maior custo de manutenção?


In [ ]:
idx_maior_custo_manut = df_base.groupby(['maquina'])['custo_manutencao'].sum().idxmax()
maior_custo_manut = df_base.groupby(['maquina'])['custo_manutencao'].sum().max()
print(f'"{idx_maior_custo_manut}" {maior_custo_manut}\n')

print(df_base.groupby(['maquina'])['custo_manutencao'].sum().sort_values(ascending=False).reset_index().head(3))



"SOLDA ROBOTIZADA 02" 14232.24

                maquina  custo_manutencao
0   SOLDA ROBOTIZADA 02          14232.24
1          EXTRUSORA 01           9450.95
2  PRENSA HIDRÁULICA 02           8809.28


##### 4. Qual máquina apresenta maior consumo de energia por unidade?


In [164]:
idx_max_consumo = df_base.groupby(['maquina'])['consumo_por_unidade'].mean().idxmax()
max_consumo = df_base.groupby(['maquina'])['consumo_por_unidade'].mean().round(6).max()
print(f'"{idx_max_consumo}" - {max_consumo}\n')

print(df_base.groupby(['maquina'])['consumo_por_unidade'].mean().sort_values(ascending=False).reset_index().head(3))

"PRENSA HIDRÁULICA 01" 0.031862

                maquina  consumo_por_unidade
0  PRENSA HIDRÁULICA 01             0.031862
1          FRESADORA 01             0.031328
2   SOLDA ROBOTIZADA 02             0.030562


##### 5. Qual produto possui maior taxa de rejeição?


In [172]:
idx_max_tx_rejeicao = df_base.groupby(['produto'])['taxa_rejeicao'].mean().idxmax()
max_tx_rejeicao = df_base.groupby(['produto'])['taxa_rejeicao'].mean().round(6).max()
print(f'"{idx_max_tx_rejeicao}" - {max_tx_rejeicao}\n')

print(df_base.groupby(['produto'])['taxa_rejeicao'].mean().sort_values(ascending=False).reset_index().head(3))

"Componente Hidráulico" - 0.06254

                 produto  taxa_rejeicao
0  Componente Hidráulico       0.062540
1       Suporte Metálico       0.062048
2       Carcaça Plástica       0.060400


##### 6. Qual turno apresenta melhor produtividade?


In [173]:
idx_turno_produt = df_base.groupby(['turno'])['quantidade_produzida'].mean().idxmax()
turno_produt = df_base.groupby(['turno'])['quantidade_produzida'].mean().round(2).max()
print(f'"{idx_turno_produt}" - {turno_produt}\n')

print(df_base.groupby(['turno'])['quantidade_produzida'].mean().sort_values(ascending=False).reset_index().head(3))

"tarde" - 461.78

   turno  quantidade_produzida
0  tarde            461.780952
1  manhã            458.250000
2  noite            428.972973


##### 7. Existe relação entre temperatura e falhas?


In [202]:
correlacao_temp_falhas = df_base.groupby(['maquina']).agg(
    media_temperatura = ('temperatura', 'mean'),
    quantidade_falhas = ('status', lambda x: (x == 'falha').sum())
)

print(correlacao_temp_falhas.sort_values('quantidade_falhas',ascending=False).reset_index().head(3))


               maquina  media_temperatura  quantidade_falhas
0  SOLDA ROBOTIZADA 02          70.927941                  2
1         EXTRUSORA 01          63.768750                  0
2         FRESADORA 01          66.222222                  0


##### 8. Existe relação entre vibração e manutenção?


In [210]:
corr_vibracao_manutencao = df_base.groupby(['maquina']).agg(
    quantidade_manutencao = ('id_manutencao', 'count'),
    media_vibracao = ('vibracao', 'mean')
)

corr_vibracao_manutencao = corr_vibracao_manutencao.reset_index()

result = corr_vibracao_manutencao['quantidade_manutencao'].corr(corr_vibracao_manutencao['media_vibracao'])

print(f'Correlação agrupada por maquina entre vibração x quantidade_manutenção_por_maquina: {result:.2f}')

print(corr_vibracao_manutencao)




Correlação agrupada por maquina entre vibração x manutenção: 0.11
                maquina  quantidade_manutencao  media_vibracao
0          EXTRUSORA 01                     11        2.143438
1          FRESADORA 01                      8        2.245556
2          FRESADORA 02                      8        2.153103
3  INJETORA PLÁSTICA 01                      7        2.275161
4  PRENSA HIDRÁULICA 01                      8        2.077187
5  PRENSA HIDRÁULICA 02                     11        2.198235
6   SOLDA ROBOTIZADA 01                     11        2.122308
7   SOLDA ROBOTIZADA 02                     10        3.369706
8          TORNO CNC 01                     10        2.185333
9          TORNO CNC 02                      8        2.246757


##### 9. Qual máquina apresenta maior tempo de parada?


In [ ]:
idx_max_tempo_parada = df_base.groupby(['maquina'])['tempo_parada'].sum().idxmax()
max_tempo_parada = df_base.groupby(['maquina'])['tempo_parada'].sum().max()

print(f'"{idx_max_tem}" - {}')